In [0]:
df = spark.table("silver_fraud_features")

pd_df = df.toPandas()

In [0]:
feature_cols = [
    "amount",
    "hour",
    "day_of_week",
    "is_night",
    "amount_zscore",
    "tx_count_user"
]

X = pd_df[feature_cols]

In [0]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination=0.02,
    random_state=42
)

model.fit(X)

In [0]:
pd_df["anomaly_pred"] = model.predict(X)

In [0]:
pd_df["is_fraud"] = (
    pd_df["anomaly_pred"] == -1
).astype(int)

In [0]:
pd_df[
    pd_df["is_fraud"] == 1
].head(20)

In [0]:
gold_fraud_pd = pd_df[
    [
        "transaction_id",
        "user_id",
        "amount",
        "country",
        "is_fraud"
    ]
]

gold_fraud_df = spark.createDataFrame(
    gold_fraud_pd
)

In [0]:
gold_fraud_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fraud_alerts")

In [0]:
display(
    spark.table("gold_fraud_alerts")
)